# 05 · Orange: reproducción visual del clustering

Este notebook **verifica el archivo de exportación** y documenta cómo reproducir
el análisis en Orange Canvas sin escribir código. Orange no se usa para producir
el resultado final (eso lo hace el pipeline en Python), sino como **control
independiente**: si un analista carga el CSV y repite el clustering, debe llegar a
una estructura parecida.

> Nota: el archivo `ferramenta_orange.ows` del proyecto anterior está en formato
> XML de **Orange 2** y no abre en Orange 3. Aquí se documenta el flujo para
> reconstruirlo en Orange 3 y se entrega el CSV listo para usar.

In [1]:
import sys
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25,
                     "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

from src import config as C

print("Raiz del proyecto:", RAIZ)
print("Ventana:", C.MES_INICIO, "a", C.MES_FIN, f"({C.N_MESES} meses)")

Raiz del proyecto: C:\Users\yennz\OneDrive\Documentos\Ciencia de Datos
Ventana: 2021-01 a 2026-08 (68 meses)


In [2]:
orange = pd.read_csv(C.ORANGE_DIR / "aneel_orange.csv")
serie = pd.read_csv(C.ORANGE_DIR / "aneel_orange_serie68.csv")
print("aneel_orange.csv:", orange.shape)
print("aneel_orange_serie68.csv:", serie.shape)
orange.head(3)

aneel_orange.csv: (700, 41)
aneel_orange_serie68.csv: (700, 72)


,unidad,agente,clase,cluster,cluster_nombre,cv,tendencia_pct_mes,tendencia_r2,amplitud_estacional,fuerza_estacionalidad,ac1,ac12,yoy_media,yoy_std,drawdown_max,rango_p90_p10,iqr_relativo,share_bajo_80,share_alto_120,crecimiento_12m,...,pico_cos,valle_sin,valle_cos,total_mwh,consumidores_medios,PC1,PC2,PC3,est_M01,est_M02,est_M03,est_M04,est_M05,est_M06,est_M07,est_M08,est_M09,est_M10,est_M11,est_M12
0,CASTRO-DIS | Comercial,CASTRO-DIS,Comercial,2,Crecimiento acelerado,0.153570,-0.020632,0.000695,0.337339,0.345960,0.582125,0.422354,0.023354,0.168364,0.491124,0.387436,0.187538,0.000000,0.147059,0.052549,...,-0.866025,5.000000e-01,0.866025,13834.031,115.602941,0.220907,-4.537783,2.978480,190.254500,187.671167,201.046833,194.843833,190.557167,191.163833,203.181500,256.300000,205.7816,219.1410,202.3558,201.5052
1,CASTRO-DIS | Iluminação pública,CASTRO-DIS,Iluminação pública,2,Crecimiento acelerado,0.109073,0.240689,0.187594,0.194303,0.260835,0.721490,0.326692,0.052157,0.136615,0.267824,0.207848,0.113827,0.029412,0.029412,0.208338,...,-0.866025,5.000000e-01,0.866025,2718.615,5.558824,3.971108,-5.843154,1.654848,37.023667,34.745833,40.040833,38.797167,42.053833,41.414167,41.989833,42.514000,41.1168,40.8036,40.0310,39.4764
2,CASTRO-DIS | Industrial,CASTRO-DIS,Industrial,0,Contracción sostenida,0.637663,-2.580032,0.630682,0.369101,0.042709,0.941821,0.610021,-0.213849,0.396902,0.976021,1.611982,1.347625,0.294118,0.602941,-0.963368,...,0.500000,1.224647e-16,-1.000000,82759.005,6.338235,-5.387808,-0.721191,2.972424,1093.704333,1107.993167,1489.929000,1464.367500,1197.868833,1049.150167,1040.717000,1062.193833,1162.3918,1255.2538,1421.9540,1305.0928


In [3]:
estructura = pd.DataFrame({
    "columna": orange.columns,
    "tipo": [str(t) for t in orange.dtypes],
    "nulos": orange.isna().sum().values,
    "unicos": [orange[c].nunique() for c in orange.columns],
})
estructura.head(20)

,columna,tipo,nulos,unicos
0,unidad,str,0,700
1,agente,str,0,101
2,clase,str,0,7
3,cluster,int64,0,3
4,cluster_nombre,str,0,3
5,cv,float64,0,700
6,tendencia_pct_mes,float64,0,700
7,tendencia_r2,float64,0,700
8,amplitud_estacional,float64,0,700
9,fuerza_estacionalidad,float64,0,700


In [4]:
print("Distribucion del cluster:")
print(orange["cluster_nombre"].value_counts().to_string())
print()
print("Clases:", orange["clase"].nunique(), "| agentes:", orange["agente"].nunique())
print("Nulos en variables de clustering:",
      int(orange[[c for c in orange.columns if c.startswith(("cv", "tendencia", "crecimiento", "yoy"))]].isna().sum().sum()))

Distribucion del cluster:
cluster_nombre
Contracción sostenida    366
Base estable             187
Crecimiento acelerado    147

Clases: 7 | agentes: 101
Nulos en variables de clustering: 0


## Flujo recomendado en Orange 3

**Opción A — réplica exacta del modelo final (recomendada)**

1. *File* → `reports/orange/aneel_orange_serie68_z.csv`.
   Esta matriz ya viene **winsorizada y estandarizada por fila**, que es el espacio
   del modelo final. No aplicar *Normalize*: Orange normaliza por columna, no por
   fila, y volver a normalizar alteraría la réplica.
2. *Select Columns* → **Target** = `cluster_nombre`; **Features** = las 68 columnas
   `2021-01` … `2026-08`; **Metas** = `unidad`, `agente`, `clase`, `cluster`.
3. *k-Means* → clusters = 3, initialization = k-means++, n_init = 10,
   random seed = 42.
4. *Distances* → Euclidean, conectado a la salida de *File*.
5. *Silhouette* → conectar *Distances* y la salida de *k-Means*.
6. *Scatter Plot* → abrir `aneel_orange.csv` en otro *File*, ejes `PC1` y `PC2`,
   color = `cluster_nombre`. Los PCs ya están calculados sobre la serie normalizada.
7. *Box Plot* → variable `tendencia_pct_mes`, agrupada por `cluster_nombre`.
8. *Data Table* → revisar `unidad`, `cluster_nombre`, `tendencia_pct_mes`.

**Opción B — control con descriptores (no recomendada para el resultado)**

1. *File* → `aneel_orange.csv`.
2. *Select Columns* → Target = `cluster_nombre`; Features = los 19 descriptores
   (`cv`, `tendencia_pct_mes`, …, `valle_cos`). Excluir `total_mwh`,
   `consumidores_medios` y `PC*`.
3. *Preprocess* → **Normalize Features** (por columna).
4. *k-Means* (k = 3) → *Silhouette* → *Scatter Plot*.

Esta opción produce un clúster de ~6 unidades: confirma que los descriptores con
normalización por columna no son una base estable para k = 3, en línea con el DoE.

**Verificación esperada (Opción A)**

| Chequeo | Valor de referencia |
|---|---|
| Tamaños de clúster | 366 / 187 / 147 |
| Silhouette (serie 68 normalizada, k = 3) | 0,2635 |
| ARI vs proyecto | 1,000 |
| Tendencia media del grupo de contracción | ≈ −1,12 %/mes |
| Distribución de clases | Industrial/Rural/Comercial en contracción |

Si Orange devuelve grupos de tamaño 700/0/0, revisar que `cluster_nombre` no esté
entre las features. Si devuelve grupos por tamaño, se está usando el CSV sin
normalizar (`aneel_orange_serie68.csv`) en lugar de `_z`.


## Alternativa programática (si Orange 3 está instalado)

Orange se puede usar desde Python. El siguiente bloque se ejecuta solo si el
paquete está disponible; de lo contrario muestra la instrucción de instalación.

In [5]:
try:
    import Orange
    print("Orange3 disponible:", Orange.__version__)
    datos = Orange.data.Table(str(C.ORANGE_DIR / "aneel_orange.csv"))
    print("Instancias:", len(datos), "| variables:", len(datos.domain))
except ImportError:
    print("Orange3 no esta instalado en este entorno.")
    print("Instalacion sugerida en un entorno aparte (Python 3.10-3.12):")
    print("  python -m venv .venv-orange")
    print("  .venv-orange\\Scripts\\activate")
    print("  pip install orange3")

Orange3 no esta instalado en este entorno.
Instalacion sugerida en un entorno aparte (Python 3.10-3.12):
  python -m venv .venv-orange
  .venv-orange\Scripts\activate
  pip install orange3


## Por qué no se fuerza Orange en el entorno principal

El proyecto corre con Python 3.15, pandas 3.0 y scikit-learn 1.9. Orange 3
todavía no publica wheels estables para esa combinación. Forzarlo produciría un
entorno frágil y no aportaría nada al resultado: el clustering ya está validado
con scikit-learn, MLflow y pruebas estadísticas. Orange queda como **herramienta
de verificación visual** en un entorno separado, con el CSV exportado y las
instrucciones de esta guía.

## Entregables para la demostración

- `reports/orange/aneel_orange.csv`: 700 filas, etiqueta + descriptores + PCs + 12 meses.
- `reports/orange/aneel_orange_serie68.csv`: 700 filas × 68 meses + metadatos.
- Este notebook: verificación de esquema y receta paso a paso.

## Verificación ejecutada con Orange 3.41

Orange se instaló en un entorno separado (Python 3.14, Orange 3.41) y se ejecutó
`tools/orange_check.py`. Resultados:

| Prueba | ARI vs proyecto | Tamaños | Lectura |
|---|---:|---|---|
| Réplica del modelo final (serie 68, winsor + z por fila) | **1,000** | 366 / 187 / 147 | el CSV y la configuración reproducen la partición exacta |
| Alternativa con 19 descriptores + Normalize por columna | 0,036 | 509 / 185 / 6 | la representación de descriptores no es viable con k=3 |

**Matiz honesto:** la réplica exacta usa el mismo KMeans de scikit-learn que el
pipeline, así que ARI = 1 valida la exportación y la configuración, no un algoritmo
distinto. La alternativa con descriptores sí es una representación diferente y
confirma lo que el DoE ya mostraba: los descriptores con normalización por columna
producen un clúster de 6 unidades y no son una base estable para k=3.


In [6]:
import json

ruta = C.TABLES_DIR / "orange_verificacion.json"
if ruta.exists():
    verif = json.loads(ruta.read_text(encoding="utf-8"))
    print("Python:", verif["python"], "| Orange:", verif["orange"])
    print("Réplica del modelo final:")
    print("  ARI =", verif["replica_modelo_final"]["ari_vs_proyecto"],
          "| tamaños =", verif["replica_modelo_final"]["tamanos"])
    print("Alternativa con descriptores:")
    print("  ARI =", verif["alternativa_descriptores"]["ari_vs_proyecto"],
          "| tamaños =", verif["alternativa_descriptores"]["tamanos"])
else:
    print("Aun no existe orange_verificacion.json; ejecutar tools/orange_check.py en el entorno Orange.")


Python: 3.14.7 | Orange: 3.41.0
Réplica del modelo final:
  ARI = 1.0 | tamaños = {'0': 366, '1': 187, '2': 147}
Alternativa con descriptores:
  ARI = 0.0358 | tamaños = {'0': 509, '1': 185, '2': 6}
